### Ingest Registered Users CSV
This notebook ingests the registered users file from the landing layer, standardizes the dataset, and writes it to a bronze Delta table.

Business context:
* This table shows the people who registered for the auction
* Each row represents a bidder registration record tied to a specific auction
* The bronze table preserves these records for downstream analysis of bidder registration, approval activity, and auction participation

#### Step 1: Load shared setup values
This step imports the common project configuration used across bronze ingestion notebooks.

What happens here:
* Loads shared Unity Catalog and volume variables
* Reuses helper functions such as metadata column enrichment
* Keeps environment-specific values centralized in one setup notebook


#### Step 2: Build the registered users file path
This step assembles the full storage path for the raw registered users CSV.

What happens here:
* Combines the configured volume, catalog, schema, and folder names
* Points to the specific dated registered users file to ingest
* Makes the resolved source path easy to validate before reading data


#### Step 3: Define the registered users schema
This step declares the expected structure of the CSV so Spark reads auction, bidder, contact, and approval fields with the intended data types.

What happens here:
* Defines the incoming registered user columns explicitly
* Applies consistent types for bidder identifiers and registration timestamps
* Avoids schema drift that can happen with CSV inference alone


#### Registered users data dictionary
This file represents bidder registration records for an auction, where each row shows a person who registered and their related contact, billing, shipping, and approval details.

Key fields:
* `Auction Name` and `Auction Date`: identify the auction the person registered for
* `Bidder number`: the bidder identifier used by the auction platform
* `First Name`, `Last Name`, `Phone Number`, and `Email`: primary contact details for the registered bidder
* Billing and shipping fields: customer address and fulfillment-related information captured during registration
* `Registration status`: the current status of the bidder registration
* `Auto Approved`: indicates whether the registration was approved automatically
* `Registration Datetime`: when the registration was submitted
* `Date approved`: when the registration was approved, if applicable

Business use:
* Track who registered for each auction
* Analyze bidder approval timing and registration outcomes
* Support downstream reporting on auction participation and bidder operations


#### Step 4: Load and enrich the registered users data
This step reads the raw CSV into Spark and appends standard metadata columns used for traceability.

What happens here:
* Loads the registered users file into a Spark DataFrame
* Applies the predefined schema during ingestion
* Adds metadata such as source path and ingestion timestamp


#### Step 5: Prepare the bronze-ready registration dataset
This step standardizes the DataFrame so the columns are easier to use after the bronze write.

What happens here:
* Renames source columns into a lowercase, underscore-based format
* Cleans special characters in address-related field names for downstream usability
* Confirms the bronze target context before persisting the table


#### Step 6: Write the bronze registered users table
This final step persists the cleaned registered users DataFrame as a Delta table in the bronze schema.

What happens here:
* Writes the transformed registration data to a managed Delta table
* Rebuilds the target table on reruns using overwrite mode
* Makes the bronze dataset available for downstream silver transformations, auction participation analysis, and approval reporting


In [ ]:
catalog = dbutils.widgets.get("catalog")
container = dbutils.widgets.get("container")
storage_account = dbutils.widgets.get("storage_account")

In [ ]:
%run "../setup/002_Environment_variables"

In [ ]:
dbutils.widgets.text('id'," ")
batch_id = dbutils.widgets.get('id')
batch_id

In [ ]:
# Build the full source path for the CSV file to ingest from cloud storage.
# This keeps the input file location parameterized through shared environment values.
path = f'/{volume}/{catalog}/{schema}/{file}/{batch_id}/registered_users_*.csv'

In [ ]:
# Import explicit Spark SQL data types so the CSV schema can be declared manually.
from pyspark.sql.types import *

# Define the expected structure of the registered users CSV.
# Using an explicit schema prevents unwanted type inference issues.
schema_registered_users = StructType([
    StructField('Platform', StringType(), True),
    StructField('Auction Name', StringType(), True),
    StructField('Auction Date', StringType(), True),
    StructField('Bidder number', IntegerType(), True),
    StructField('First Name', StringType(), True),
    StructField('Last Name', StringType(), True),
    StructField('Phone Number', StringType(), True),
    StructField('Email', StringType(), True),
    StructField('Billing Company', StringType(), True),
    StructField('Billing First name', StringType(), True),
    StructField('Billing Last name', StringType(), True),
    StructField('Billing Phone Number', StringType(), True),
    StructField('Billing Address 1', StringType(), True),
    StructField('Billing Address 2', StringType(), True),
    StructField('Billing City', StringType(), True),
    StructField('Billing State/Province', StringType(), True),
    StructField('Billing ZIP/Postal Code', StringType(), True),
    StructField('Billing Country', StringType(), True),
    StructField('Shipping Company', StringType(), True),
    StructField('Shipping First name', StringType(), True),
    StructField('Shipping Last name', StringType(), True),
    StructField('Shipping Phone Number', StringType(), True),
    StructField('Shipping Address 1', StringType(), True),
    StructField('Shipping Address 2', StringType(), True),
    StructField('Shipping City', StringType(), True),
    StructField('Shipping State/Province', StringType(), True),
    StructField('Shipping ZIP/Postal Code', StringType(), True),
    StructField('Shipping Country', StringType(), True),
    StructField('Registration status', StringType(), True),
    StructField('Auto Approved', StringType(), True),
    StructField('Registration Datetime', TimestampType(), True),
    StructField('Date approved', TimestampType(), True)
])

In [ ]:
# Read the source CSV into a Spark DataFrame using the predefined schema.
# The file is expected to contain a header row.
df_registered_users= spark.read.format('csv') \
    .option('header', 'true') \
         .option('multiLine', 'true') \
    .option("quote", '"') \
    .option("escape", '"') \
    .schema(schema_registered_users) \
    .load(path)


# Add standard metadata or audit columns using the shared helper function.
df_registered_users = add_columns(df_registered_users)

In [ ]:
df_registered_users = df_registered_users.withColumnsRenamed({col: col.replace(' ', '_').lower() for col in df_registered_users.columns})

In [ ]:
df_registered_users = df_registered_users.withColumnsRenamed({'billing_state/province': 'billing_state_province', 'billing_zip/postal_code': 'billing_zip_postal_code','shipping_state/province':'shipping_state_province', 'shipping_zip/postal_code': 'shipping_zip_postal_code'})

In [ ]:
# Preview the target bronze namespace components before writing the Delta table.
f'{volume}/{catalog}/{bronze}'

In [ ]:
# Write the ingested DataFrame to the bronze layer as a Delta table.
# Overwrite mode is used so the table can be rebuilt on repeated runs.
df_registered_users \
    .write \
    .format('delta') \
    .mode('overwrite') \
    .partitionBy('batch_id') \
    .option('replaceWhere',f"batch_id = '{batch_id}'")\
    .saveAsTable(f'{catalog}.{bronze}.registeredusers')